# Lab 2: Connecting External MCP Sources

In Lab 1, you built your own MCP server. Now you'll connect to **existing** MCP servers created by the community.

This is the power of MCP as an ecosystem — hundreds of pre-built integrations you can plug into any AI client.

```
┌──────────────┐
│  AI Client   │──── Your dev-tools server (Lab 1)
│ (Claude Code │──── Filesystem MCP (read/write files)
│  / VS Code)  │──── GitHub MCP (repos, issues, PRs)
│              │──── Fetch MCP (web pages)
└──────────────┘
```

All running simultaneously, all accessible through the same AI interface.

---
## Part 1: Finding MCP Servers

### Where to look

| Source | URL | What's there |
|--------|-----|------|
| Official Reference Servers | [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers) | Maintained by the MCP team |
| Awesome MCP Servers | [github.com/wong2/awesome-mcp-servers](https://github.com/wong2/awesome-mcp-servers) | Community curated list |
| MCP Servers Directory | [mcpservers.org](https://mcpservers.org) | Searchable directory |

### Evaluating a server

Before connecting any MCP server, check:

1. **Source code** — Is it open source? Can you read what it does?
2. **Maintainer** — Who maintains it? Is it actively updated?
3. **Permissions** — What does it access? Files? Network? APIs?
4. **Transport** — STDIO (local) or HTTP (remote)?
5. **Dependencies** — Does it need API keys? Docker? Other services?

---
## Part 2: Filesystem MCP Server

The **Filesystem MCP** server provides file read/write/search operations through MCP.

### Install and test

```bash
# Test it directly (replace /path/to/dir with a directory you want to expose)
npx -y @modelcontextprotocol/server-filesystem /path/to/dir
```

### Available tools

| Tool | What it does |
|------|------------|
| `read_file` | Read contents of a file |
| `write_file` | Write content to a file |
| `list_directory` | List files in a directory |
| `search_files` | Search for files matching a pattern |
| `get_file_info` | Get file metadata (size, dates) |

### Configuration

```json
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-filesystem", "."]
    }
  }
}
```

> **Security**: The path argument (`.` above) controls which directory the server can access. Only expose directories you're comfortable with.

---
## Part 3: GitHub MCP Server

The **GitHub MCP** server provides full GitHub API access — search repos, read files, manage issues and PRs.

### Prerequisites

You need a GitHub Personal Access Token:
1. Go to https://github.com/settings/tokens
2. Generate a new token with `repo` scope
3. Set it as an environment variable:

```bash
export GITHUB_TOKEN=ghp_your_token_here
```

### Configuration

```json
{
  "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": "<YOUR_TOKEN>"
      }
    }
  }
}
```

### Try it

After configuring, ask Claude:
- *"Search GitHub for MCP server implementations in Python"*
- *"List the open issues on <your-repo>"*
- *"Read the README.md from <some-repo>"*

---
## Part 4: Fetch MCP Server

The **Fetch MCP** server lets AI assistants fetch web pages and convert them to markdown.

### Configuration

```json
{
  "mcpServers": {
    "fetch": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-fetch"]
    }
  }
}
```

### Try it

- *"Fetch the MCP documentation from modelcontextprotocol.io and summarize it"*
- *"What's on the front page of Hacker News right now?"*

---
## Part 5: Combining Multiple Servers

The real power comes from combining servers. One config file can connect multiple MCP servers simultaneously.

### Full combined config

See `configs/mcp_external.json` for a ready-to-use config that combines all three external servers.

```bash
# For Claude Code:
cp labs/lab2-external-mcp-sources/configs/mcp_external.json .mcp.json

# For VS Code:
cp labs/lab2-external-mcp-sources/configs/vscode_mcp_external.json .vscode/mcp.json
```

### Combining with Lab 1

You can also add your own Lab 1 server alongside the external ones. Just add it to the same JSON:

```json
{
  "mcpServers": {
    "dev-tools": {
      "command": "python",
      "args": ["labs/lab1-local-mcp-servers/servers/dev_tools_stdio.py"]
    },
    "filesystem": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-filesystem", "."]
    },
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": { "GITHUB_PERSONAL_ACCESS_TOKEN": "<TOKEN>" }
    }
  }
}
```

Now your AI assistant has access to your custom tools, file operations, AND GitHub — all at once.

---
## Part 6: Security Considerations

### Trust Boundaries

```
┌─────────────────────────────────────────────┐
│  YOUR MACHINE (trust boundary)              │
│                                             │
│  ┌─────────┐     ┌───────────────────┐      │
│  │ AI Chat │────►│ MCP Server        │      │
│  │         │     │ (runs YOUR code)  │      │
│  └─────────┘     └───────┬───────────┘      │
│                          │                  │
│                  ┌───────┴───────────┐      │
│                  │ Local Resources   │      │
│                  │ Files, Ports, SSH │      │
│                  └───────────────────┘      │
└─────────────────────────────────────────────┘
```

### Key principles

1. **Least privilege**: Only expose the directories/APIs you need
2. **No hardcoded secrets**: Use environment variables for tokens
3. **Review before connecting**: Read the server's source code
4. **Monitor access**: Check what tools are being called (FastMCP logging)
5. **Version pinning**: Use specific versions in production

---
## Summary

| Concept | What You Learned |
|---------|------------------|
| External MCP servers | Pre-built tools you can plug in via config |
| Filesystem MCP | File operations through MCP |
| GitHub MCP | GitHub API access through MCP |
| Fetch MCP | Web page fetching through MCP |
| Multi-server config | Combine multiple MCP servers in one config |
| Security | Trust boundaries, least privilege, secret management |

**Next**: [Lab 3 — Azure Deployment](../lab3-azure-deployment/README.md)